In [ ]:
# 1️⃣ Install required packages
!pip install -q langchain==0.3.27 langchain-community langchain-huggingface transformers torch sentence-transformers faiss-cpu

# 2️⃣ Imports
import time
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import pipeline
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader

# 3️⃣ Load your document (upload DSML.txt in Colab)

loader = TextLoader("DSML.txt")
docs = loader.load()

# 4️⃣ Split into chunks

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(docs)

# 5️⃣ Create embeddings + FAISS

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)

# 6️⃣ Load Phi-2 model

generator = pipeline("text-generation", model="microsoft/phi-2", max_new_tokens=256, do_sample=False)

# 7️⃣ Helper: Extract clean text

def extract_generated_text(raw_output, prompt):
    text = raw_output.replace(prompt, "").strip()
    lines = text.split("\n")
    clean = []
    for line in lines:
        line_strip = line.strip()
        if (
            not line_strip
            or line_strip.lower().startswith("answer:")
            or "you are a helpful assistant" in line_strip.lower()
            or line_strip.startswith("--------------------")
            or line_strip.lower().startswith("question:")
            or line_strip.lower().startswith("solution:")
        ):
            continue
        clean.append(line_strip)
    return " ".join(clean).strip()

# 8️⃣ RAG Response (STRICT MODE)

def rag_response(query, k=3):
    start = time.time()
    retrieved_docs = vectorstore.similarity_search(query, k=k)
    if not retrieved_docs:
        return "I don’t know the answer.", round(time.time() - start, 2)

    context = " ".join([d.page_content for d in retrieved_docs])

    # If similarity is too weak → skip answering
    if len(context.strip()) < 50:
        return "I don’t know the answer.", round(time.time() - start, 2)

    # Check if query terms appear in context — adds stricter filtering
    keywords = [word.lower() for word in query.split() if len(word) > 3]
    match_count = sum(1 for kw in keywords if kw in context.lower())
    if match_count == 0:
        return "I don’t know the answer.", round(time.time() - start, 2)

    # Build prompt
    answer_prompt = f"""
Answer the user's question using ONLY the context below.
If the context does NOT clearly contain the answer, reply exactly: "I don’t know the answer".
Be concise and factual.

Context:
{context}

Question: {query}

Answer:
"""
    output = generator(answer_prompt, max_new_tokens=256, do_sample=False)
    raw_text = output[0]["generated_text"]
    answer = extract_generated_text(raw_text, answer_prompt)

    # If model still generates unrelated text → fallback
    if not answer or "pm of india" in answer.lower() or len(answer.split()) < 3:
        answer = "I don’t know the answer."

    end = time.time()
    return f"{answer}\n_(Response time: {round(end - start, 2)} sec)_"

# 🔟 Chat Loop
print("✅ Strict Phi-2 RAG Chatbot ready! Type 'exit' to quit.\n")

while True:
    query = input("You: ")
    if query.lower() == "exit":
        print("Bot: Session ended. Goodbye!")
        break
    response = rag_response(query)
    print(f"Bot: {response}\n")


In [ ]:
with open("requirements.txt", "w") as f:
    f.write("""\
langchain==0.3.27
langchain-community
langchain-huggingface
transformers
torch
sentence-transformers
faiss-cpu
accelerate
""")


In [ ]:
!cat requirements.txt


In [ ]:
!zip -r RAG_Based_Chatbot.zip rag_chatbot.ipynb DSML.txt requirements.txt
from google.colab import files
files.download("RAG_Based_Chatbot.zip")

